**Privacy note:** Notebook outputs were cleared to avoid exposing dataset records. Configure source paths privately using environment variables.

#### Online Shopping Purchase Intent



##### Business Problem
##### Purchase behaviour was hidden in noisy, high-volume browsing sessions, making conversion intent difficult to detect in real time

##### Objective
##### Predict whether a session ends with a purchase completed by understanding behavior patterns and data quality constraints.

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import lightgbm
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


#### Dataset Source

In [ ]:
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)

TARGET = 'PurchaseCompleted'
df = pd.read_csv(os.environ["SHOPPING_DATA_PATH"], na_values=['NULL', 'nan', 'None'], keep_default_na=True)

#### Data Types

In [ ]:
print('Dataset shape:', df.shape)
print('\nColumns:', list(df.columns))
print('\nData types:\n', df.dtypes)
df.head()

#### Feature Schema Reference

#### Numeric Features (Bounded Rates)
- **BounceRate** (0.0-1.0): Percentage of single-page sessions on the website
- **ExitRate** (0.0-1.0): Percentage of users who exit from a specific page after viewing it
  - Formula: (Number of exits from a page) / (Total views of that page)
- **SpecialDayProximity** (0.0-1.0): Proximity score to special events *(NOT a day count)*
  - 0.0 = not close to special day
  - 0.2 = slightly close
  - 0.4 = moderately close
  - 0.6 = getting closer
  - 0.8 = very close
  - 1.0 = the special day itself

#### Numeric Features
- **PageValue**: Page contribution to revenue. Verify that this value is available before the prediction decision; if it uses revenue or completed-transaction information, it is target leakage and must not be used for real-time scoring.
- **ProductPageTime**: Time spent on product pages in seconds. Inspect the observed range and skew from the dataset rather than hard-coding a range in the documentation.

#### Categorical Features
- **TrafficSource**: Numeric codes representing traffic channels; treated as categorical, not ordinal. The observed category set is checked below.
- **GeographicRegion**: Numeric codes for geographic regions; unknown or special codes are retained as categories.
- **CustomerType**: Visitor classification, normalized to lowercase by the shared preprocessing function.

#### Target
- **PurchaseCompleted** (0/1): Binary indicator of whether the session resulted in a purchase

#### Data Quality: Percentage of Missing Values

In [ ]:
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False).round(2)
print('Missing percentage by feature:\n', missing_pct)

#### Data Quality: Inspect invalid counts

In [ ]:
invalid_counts_br = ((df['BounceRate'] < 0) | (df['BounceRate'] > 1)).sum()
invalid_counts_sdp = ((df['SpecialDayProximity'] < 0) | (df['SpecialDayProximity'] > 1)).sum()
invalid_counts_er = ((df['ExitRate'] < 0) | (df['ExitRate'] > 1)).sum()

print('Bounce Rate Invalid Counts:', invalid_counts_br)
print('Special Day Proximity Invalid Counts:', invalid_counts_sdp)
print('Exit Rate Invalid Counts:', invalid_counts_er)

#### Data Quality: Duplicated Rows Counts

In [ ]:
duplicated_counts = df.duplicated().sum()
duplicated_counts

#### Convert less than 0 and more than 1 values into Null values

In [ ]:
df = domain_clean(df)

print('Missing values after cleanup (%):\n')
print((df.isna().mean() * 100).round(2).sort_values(ascending=False))

#### Missing Values after cleanup

In [ ]:
print('\nMissing values after cleanup (%):\n')
print((df.isna().mean() * 100).round(2).sort_values(ascending=False))

### Exploratory Data Analysis

#### Imbalanced Class Distribution

In [ ]:
class_dist = df[TARGET].value_counts().sort_index().astype(str)
class_pct = (df[TARGET].value_counts(normalize=True).sort_index() * 100).round(2).astype(str)
print('\nClass distribution (count):\n', class_dist)
print('\nClass distribution (%):\n', class_pct)

#### Target Class Distribution

In [ ]:
bars = plt.bar(class_dist.index.astype(str), class_dist.values.astype(int))
plt.title('Target Class Distribution')
plt.xlabel('Purchase Completed')
plt.ylabel('Session Count')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(class_dist.index.astype(str), class_dist.values)
plt.title('Target Class Distribution')
plt.ylabel('Session Count')
plt.tight_layout()
plt.show()

#### The purpose is to determine whether sessions with a positive PageValue have a higher conversion rate than the overall baseline conversion rate

In [ ]:
page_value_positive = df['PageValue'] > 0
page_value_positive.value_counts()

page_value_conversion = (df.assign(PageValuePositive=page_value_positive).groupby('PageValuePositive')[TARGET].agg(['count', 'sum', 'mean']).rename(columns={'count': 'sessions', 'sum': 'purchases', 'mean': 'conversion_rate'}))
page_value_conversion

#### 25% of the sessions spent around 2929 seconds on product pages

In [ ]:
high_time_cut = np.nanpercentile(df['ProductPageTime'], 75).round(2)
high_time_sessions = df['ProductPageTime'] >= high_time_cut
high_time_cut, high_time_sessions.sum()


#### 2940 sessions had an exit rate of 0.45% or lower, indicating that pages visited in these sessions relatively low likelihood of being visitor's final page before leaving the website

In [ ]:
low_exit_cut = np.nanpercentile(df['ExitRate'], 25).round(4)
low_exit_sessions = df['ExitRate'] <= low_exit_cut
low_exit_cut, low_exit_sessions.sum()


#### 2433 Sessions had a bounce rate of 0%, meaning no bounce behaviour was recorded for those sessions

In [ ]:
low_bounce_cut = np.nanpercentile(df['BounceRate'], 25).round(4)
low_bounce_cut_sessions = df['BounceRate'] <= low_bounce_cut
low_bounce_cut, low_bounce_cut_sessions.sum()

In [ ]:
num_cols = ['SpecialDayProximity', 'ExitRate', 'PageValue', 'BounceRate', 'ProductPageTime']

mean_by_target = df.groupby(TARGET)[num_cols].mean()
print(mean_by_target)

### Interpretation: Numeric Drivers
- Purchasing sessions generally show higher `PageValue` and longer `ProductPageTime`, indicating stronger buying intent.
- `ExitRate` and `BounceRate` are lower for purchase sessions, suggesting deeper and more engaged browsing behavior.
- These patterns support using value/engagement metrics as primary predictive features.

In [ ]:
customer_conv = (df.groupby('CustomerType', dropna=False)[TARGET].mean().sort_values(ascending=False))
customer_conv

In [ ]:
traffic_conv = (df.groupby('TrafficSource', dropna=False)[TARGET].mean().sort_values(ascending=False))
traffic_conv

In [ ]:
region_conv = (df.groupby('GeographicRegion', dropna=False)[TARGET].mean().sort_values(ascending=False))
region_conv

##### Customer Type

In [ ]:
customer_conv = customer_conv.sort_values(ascending=False)
plt.figure(figsize=(7,4))

bars = plt.bar(customer_conv.index.astype(str), customer_conv.values)
plt.title('Conversion Rate by Customer Type')
plt.xlabel('Customer Type')
plt.ylabel('Conversion Rate')
plt.show()

##### Traffic Source

In [ ]:
traffic_conv = traffic_conv.sort_values(ascending=False)
plt.figure(figsize=(12, 4))
bars = plt.bar(traffic_conv.index.astype(str), traffic_conv.values)
plt.title('Conversion Rate by Traffic Source')
plt.xlabel('Traffic Source')
plt.ylabel('Conversion Rate')
plt.tight_layout()
plt.show()

##### Geographic Region

In [ ]:
region_conv = region_conv.sort_values(ascending=False)

plt.figure(figsize=(8, 4))

bars = plt.bar(region_conv.index.astype(str), region_conv.values)
plt.title('Conversion Rate by Geographic Region')
plt.xlabel('Geographic Region')
plt.ylabel('Conversion Rate')
plt.tight_layout()
plt.show()

In [ ]:
corr_cols = ['SpecialDayProximity', 'ExitRate', 'PageValue', 'BounceRate', 'ProductPageTime', TARGET]
corr = df[corr_cols].corr(numeric_only=True)
corr

##### Correlation analysis: Pagevalue showed the highest correlation against purchase completed while PurchaseCompleted and ProductPageTime scored the second highest in terms of correlation. However, ExitRate and PurchaseCompleted showed a weak negative correlation suggesting that sessions with higher exit rates tend to be less associated with completed purchases. No strong correlations between predictor variables, suggesting limited multicollinearity among the numerical features. 

## Final Conclusions and Implications

### Key Findings
1. The target is imbalanced, so recall/F1 and threshold tuning are necessary for fair model evaluation.
2. `PageValue` and `ProductPageTime` are strong positive signals of purchase intent.
3. `ExitRate` and `BounceRate` show negative relationships with purchase completion.
4. Conversion rates vary by `CustomerType`, `TrafficSource`, and `GeographicRegion`.

### Impact on Modeling Choices
- Apply domain-aware cleaning for bounded features.
- Treat coded source/region fields as categories.
- Use stratified splits and compare multiple models.
- Tune decision threshold for the business objective.

### Business Implications
- Prioritize users with high page value and high product engagement.
- Reduce exit/bounce behavior on high-intent pages through UX interventions.
- Allocate campaigns by high-converting traffic and region segments.